# Delete negative secondary values in a configuration

In [1]:
import os
from pathlib import Path
import tempfile
import shutil

import teehr
from teehr.models.filters import TableFilter

from pyspark.sql import functions as F
from pyspark.sql.window import Window

teehr.__version__

'0.6.2'

In [2]:
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session(
    # start_spark_cluster=True,
    # executor_instances=64,
    # executor_memory="32g",
    # executor_cores=4,
    aws_profile="default",
    # update_configs={
    #     "spark.sql.shuffle.partitions": 512,
    # }
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS credentials from ~/.aws/credentials profile 'default
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [3]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [13]:
filters = [
    TableFilter(
        column='configuration_name',
        operator='=',
        value="nwpsrfc_streamflow_forecast"
    ),
    TableFilter(
        column='value',
        operator='<',
        value=-1000
    )
]

In [14]:
# Optional preview of rows that will be deleted
rows_to_delete = ev.secondary_timeseries.delete(
    filters=filters,
    dry_run=True,
)
print(f"rows to delete: {rows_to_delete.count()}")

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


rows to delete: 212646


In [15]:
rows_to_delete.show()

+-------------------+----------+---------+-------------+------+--------------------+-------------------+-------------------+--------------------+--------------------+
|         value_time|     value|unit_name|  location_id|member|  configuration_name|      variable_name|     reference_time|          created_at|          updated_at|
+-------------------+----------+---------+-------------+------+--------------------+-------------------+-------------------+--------------------+--------------------+
|2026-04-23 12:00:00|-28288.482|    m^3/s|nwpsrfc-mcgi4|  NULL|nwpsrfc_streamflo...|streamflow_6hr_inst|2026-04-23 00:00:00|2026-04-23 01:19:...|2026-04-23 01:19:...|
|2026-04-23 18:00:00|-28288.482|    m^3/s|nwpsrfc-mcgi4|  NULL|nwpsrfc_streamflo...|streamflow_6hr_inst|2026-04-23 00:00:00|2026-04-23 01:19:...|2026-04-23 01:19:...|
|2026-04-24 18:00:00|-28288.482|    m^3/s|nwpsrfc-mcgi4|  NULL|nwpsrfc_streamflo...|streamflow_6hr_inst|2026-04-23 00:00:00|2026-04-23 01:19:...|2026-04-23 01:19:...

In [16]:
# Delete the bad slice
deleted_count = ev.secondary_timeseries.delete(filters=filters)
print(f"deleted rows: {deleted_count}")

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


deleted rows: 212646


In [17]:
spark.stop()